In [0]:
import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DoubleType

def log_pipeline_run(pipeline_name, task_name, started_at, completed_at,
                      records_read, records_written, records_rejected,
                      status, error_message=None):
    run_id = str(uuid.uuid4())
    duration_seconds = (completed_at - started_at).total_seconds()
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("task_name", StringType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("completed_at", TimestampType(), True),
        StructField("records_read", LongType(), True),
        StructField("records_written", LongType(), True),
        StructField("records_rejected", LongType(), True),
        StructField("duration_seconds", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    row = [(run_id, pipeline_name, task_name, started_at, completed_at,
            records_read, records_written, records_rejected,
            duration_seconds, status, error_message)]
    df = spark.createDataFrame(row, schema=schema)
    df.write.format("delta").mode("append").saveAsTable("urban_mobility.monitoring.pipeline_runs")
    return run_id

def log_streaming_metrics(query, pipeline_name):
    progress = query.lastProgress
    if progress is None:
        return
    schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("batch_id", LongType(), True),
        StructField("input_rows_per_second", DoubleType(), True),
        StructField("processed_rows_per_second", DoubleType(), True),
        StructField("batch_duration_ms", LongType(), True),
        StructField("num_input_rows", LongType(), True),
        StructField("late_events", LongType(), True),
        StructField("duplicates_removed", LongType(), True),
        StructField("recorded_at", TimestampType(), True),
    ])
    row = [(pipeline_name, progress.get("batchId"), progress.get("inputRowsPerSecond"),
            progress.get("processedRowsPerSecond"), progress.get("durationMs", {}).get("triggerExecution"),
            progress.get("numInputRows"), None, None, datetime.now())]
    df = spark.createDataFrame(row, schema=schema)
    df.write.format("delta").mode("append").saveAsTable("urban_mobility.monitoring.streaming_metrics")

In [0]:
from datetime import datetime

checkpoint_path = "/Volumes/urban_mobility/bronze/checkpoints/trip_events/"
schema_path = "/Volumes/urban_mobility/bronze/schemas/trip_events/"
landing_path = "/Volumes/urban_mobility/bronze/landing/"

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

from pyspark.sql import functions as F

bronze_stream = (
    raw_stream
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

_started = datetime.now()

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("urban_mobility.bronze.trip_events")
)

query.awaitTermination()
_completed = datetime.now()

result = spark.table("urban_mobility.bronze.trip_events")
_row_count = result.count()

log_streaming_metrics(query, "ingest_trip_events")
log_pipeline_run(
    pipeline_name="bronze_layer",
    task_name="ingest_trip_events",
    started_at=_started,
    completed_at=_completed,
    records_read=_row_count,
    records_written=_row_count,
    records_rejected=0,
    status="SUCCESS"
)

print("bronze.trip_events rows:", _row_count)
result.printSchema()